In [1]:
import sys
print(sys.executable)

C:\Users\mosab\anaconda3\envs\tf\python.exe


In [2]:
# from transformers import AutoConfig
# # Load the model configuration 
# config = AutoConfig.from_pretrained('moussaKam/AraBART') 
# # Get the maximum input length (max_position_embeddings) 
# max_length = config.max_position_embeddings 
# print("Maximum Length:", max_length)
# print(config)

In [3]:
print('hello')

hello


In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset
from sklearn.metrics import classification_report
from transformers import AutoConfig, AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback
import pickle
import gc


# Load the MAQA dataset
data = pd.read_csv(r'./Datasets/Medical/MAQA/MAQA_processed.csv', encoding='utf-16', sep='\t')

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\utils\hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [5]:
# Split the dataset into train (70%), validation (20%), and test (10%) while keeping category proportions
train_df, temp_df = train_test_split(data, test_size=0.3, stratify=data['category'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.33, stratify=temp_df['category'], random_state=42)

In [6]:
train_df.head()

,question,answer,category
132635,سلم علي دكتور حبه سلك قدم دور شهر نزل دم حمر ي...,ان شكل ولون الدم النازل لا يهم فمهما كان شكل ا...,امراض المسالك البولية والتناسلية
134812,انه اعا حسس قصب هئي ودئ ماختنق اهو علج,يجب ان تراجع طبيب باطني صدريه حيث ان حساسيه ال...,انف اذن وحنجرة
300971,لهب عصب بصر ولم يشف وله كثر شهر ايش لحل,ينصح بمراجعه العياده او طبيب الاعصاب هناك اسبا...,امراض العيون
157963,سلم علي رحم الل وبر ارد ان احج عام ارد اخر نزل...,حسب طريقه الاخذ تؤخذ حبتين قبل 5 ايام من الدور...,امراض نسائية
152699,سلم علي دعب رضع صدر زوج زوج سبب رفع هرمو حلب شكر,ممكن ان يحصل مثل هذا,امراض نسائية


In [7]:
val_df.head()

,question,answer,category
89803,سلم علي عمر 15 ابي صاب سكر عمر 53 كشف مرض عمر ...,يوجد احتمال للاصابه بالسكري لكن ليس بالضروره ل...,امراض باطنية
65831,سلم علي اعا فقد شهه اقل كمه طعم شعر شبع ادي ال...,انصح بزياره طبيب عام اولا لاخذ التاريخ المرضي ...,امراض باطنية
353796,لده ربع ورم عظم راس الي كثرم طبب اشع وشع رنن و...,مادام قد طمانك المختص فلا داعي للقلق الرنين به...,امراض العضلات والعظام و المفاصل
100306,سلم علي يصب خدر طقه فخذ متد حتي ركب وهذ حله جد...,يجب تفصيل الاعراض يبدو انك تعاني من الم العصب ...,امراض باطنية
222954,انت ضطراب دور شهر نظم ولن تغر زلل شوه سمر ابع ...,هناك عده اسباب لعدم انتظام الدوره الشهريه ومنه...,الامراض الجنسية


In [8]:
max_length = 512

def tokenize_function_train(examples):
    input_texts = []
    target_texts = []
    for question, answer, category in zip(examples['question'], examples['answer'], examples['category']):
        #input_texts.append(f"category: {category} question: {question}")
        input_texts.append(question)
        target_texts.append(answer)
    model_inputs = tokenizer(input_texts, max_length=max_length, truncation=True, padding="max_length")  #
    labels = tokenizer(target_texts, max_length=max_length, truncation=True, padding="max_length").input_ids  #
    model_inputs["labels"] = labels
    return model_inputs

def tokenize_function_test(examples):
    input_texts = []
    #target_texts = []
    for question, category in zip(examples['question'], examples['category']): #examples['predicted_category']
        #input_texts.append(f"category: {category} question: {question}")
        input_texts.append(question)
    model_inputs = tokenizer(input_texts, max_length=max_length, truncation=True, padding="max_length")  #
    return model_inputs

In [9]:
!set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [10]:
bert_model_name = "aubmindlab/bert-base-arabertv2"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

# ----- Configure the Model with Desired Dropout Rates -----
bert_model_config = AutoConfig.from_pretrained(
    bert_model_name,
    dropout=0.1,              # default dropout rate; change if you need more regularization
    attention_dropout=0.1,
    activation_dropout=0.1
)

#EMBED_DIM = 1024 #bert-large-arabertv2 has a hidden size of 1024.
EMBED_DIM = 768 #bert-base-arabertv2 has a hidden size of 768.
HIDDEN_DIM = 512
NUM_LAYERS = 2



# Define Model
class AraBERT_BiLSTM(nn.Module):
    def __init__(self, bert_model_name, bert_model_config, embed_dim, hidden_dim, num_layers, vocab_size):
        super(AraBERT_BiLSTM, self).__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name, bert_model_config)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size) # hidden_dim * 2 because LSTM is bidirectional

    def forward(self, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        lstm_out, _ = self.lstm(encoder_outputs.last_hidden_state)
        logits = self.fc(lstm_out)

        #During training, labels are provided (so we calculate the loss).
        #During inference, labels are not provided (we only return predictions).
        #When training, we need to compute the loss for backpropagation.
        #When doing inference (prediction), we don’t need loss—only logits.
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)#We don’t want to compute loss on padding tokens since they don’t carry meaningful information.
            #CrossEntropyLoss expects input with shape (N, C), where:
            #N = total number of tokens in the batch (batch_size * sequence_length).
            #C = number of classes (vocab_size).
            #Reshaping
            #logits.view(-1, logits.size(-1)) transforms:
            #(batch_size, sequence_length, vocab_size)  -->  (batch_size * sequence_length, vocab_size)
            #labels.view(-1) transforms:
            #(batch_size, sequence_length)  -->  (batch_size * sequence_length)
            #Now, the shapes are compatible for CrossEntropyLoss.
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {"loss": loss, "logits": logits}  # Return loss explicitly
    
        return {"logits": logits}  # Standard return

# Initialize Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = tokenizer.vocab_size
model = AraBERT_BiLSTM(bert_model_name, bert_model_config, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, vocab_size).to(device)

print(device)

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


cuda


In [11]:
# # Assume you already have train_df with a "question" column and a tokenizer loaded
# input_texts = data["question"].tolist()

# # Tokenize without truncation or padding to get the original token lengths
# temp_inputs = tokenizer(input_texts, truncation=False, padding=False)
# input_lengths = [len(ids) for ids in temp_inputs["input_ids"]]
# max_input_length = max(input_lengths)
# print("Maximum input length (in tokens):", max_input_length)
# #Maximum input length (in tokens): 138

# target_texts = data["answer"].tolist()
# temp_outputs = tokenizer(target_texts, truncation=False, padding=False)
# output_lengths = [len(ids) for ids in temp_outputs["input_ids"]]
# max_output_length = max(output_lengths)
# print("Maximum output length (in tokens):", max_output_length)
# #Maximum output length (in tokens): 171

In [12]:
# import numpy as np
# print("argmax:", data["question"][np.argmax(input_lengths)])

In [13]:
# import numpy as np
# print("argmax:", data["answer"][np.argmax(output_lengths)])

In [14]:
train_dataset = Dataset.from_pandas(train_df).map(tokenize_function_train, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_function_train, batched=True)

Map:   0%|          | 0/304180 [00:00<?, ? examples/s]

Map:   0%|          | 0/87343 [00:00<?, ? examples/s]

In [15]:
BATCH_SIZE = 32
EPOCHS = 10
# Define Training Arguments
training_args = TrainingArguments(
    output_dir='./checkpoints/AraBERT_BiLSTM/Basic_with_preprocessing/',
    learning_rate=5e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,              # With more memory, no need for accumulation
    fp16=True,                                  # Use mixed precision for faster training and less memory usage
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=500,
    logging_dir='./models/AraBERT_BiLSTM/Basic_with_preprocessing/logs/',
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
)

# ----- Initialize the Trainer with Early Stopping -----
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [16]:
from datetime import datetime
now = datetime.now()
print(now.time())

16:54:13.426098


In [ ]:
# ----- Train the Model -----
try:
    trainer.train(resume_from_checkpoint=True)
except Exception as e:
    print(f"Failed to resume from checkpoint due to: {e}")
    print("Starting training from scratch...")
    trainer.train()


#trainer.train(resume_from_checkpoint=True)
#trainer.train()

# Save the fine-tuned model
tokenizer.save_pretrained("./models/AraBERT_BiLSTM/Basic_with_preprocessing/model/")
model.save_pretrained("./models/AraBERT_BiLSTM/Basic_with_preprocessing/model/")

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\trainer.py:3017: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(checkpoint, OPTIMIZER_

Epoch,Training Loss,Validation Loss
2,6.359900,6.315878


In [ ]:
now = datetime.now()
print(now.time())

In [ ]:
import matplotlib.pyplot as plt
# ----- Extract Logged Losses -----
logs = trainer.state.log_history
# Extract training losses (logged with key "loss") and eval losses (logged with key "eval_loss")
train_losses = [log["loss"] for log in logs if "loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

# Assume one log entry per epoch; create an epoch axis accordingly.
epochs = list(range(1, len(train_losses) + 1))

# ----- Visualize the Losses -----
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, marker='o', label="Training Loss")
plt.plot(epochs, eval_losses, marker='o', label="Evaluation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Evaluation Loss over Epochs")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from tqdm import tqdm  # For progress tracking
from torch.utils.data import DataLoader


train_dataset = Dataset.from_pandas(train_df).map(tokenize_function_test, batched=True)

# Convert the Hugging Face Dataset to a DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)  # Adjust batch size as needed

# Explicitly set dataset format to PyTorch tensors
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Move the model to GPU
model.to(torch.device('cuda:0'))

# Generate predictions for each batch in the DataLoader with a progress bar
train_predictions_list = []
for batch in tqdm(train_loader, desc="Generating Predictions"):
    # Move the input tensors to the same device as the model
    input_ids = batch['input_ids'].to(torch.device('cuda:0'))
    attention_mask = batch['attention_mask'].to(torch.device('cuda:0'))

    # Generate predictions
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_length, 
                                 pad_token_id=tokenizer.pad_token_id  # Required for batch processing
                                )
    predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    train_predictions_list.extend(predictions)  # Collect predictions for all batches

In [ ]:
from tqdm import tqdm  # For progress tracking
from torch.utils.data import DataLoader


test_dataset = Dataset.from_pandas(test_df).map(tokenize_function_test, batched=True)

# Convert the Hugging Face Dataset to a DataLoader
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)  # Adjust batch size as needed

# Explicitly set dataset format to PyTorch tensors
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Move the model to GPU
model.to(torch.device('cuda:0'))

# Generate predictions for each batch in the DataLoader with a progress bar
test_predictions_list = []
for batch in tqdm(test_loader, desc="Generating Predictions"):
    # Move the input tensors to the same device as the model
    input_ids = batch['input_ids'].to(torch.device('cuda:0'))
    attention_mask = batch['attention_mask'].to(torch.device('cuda:0'))

    # Generate predictions
    with torch.no_grad():        
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_length, 
                                 pad_token_id=tokenizer.pad_token_id  # Required for batch processing
                                )
    predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    test_predictions_list.extend(predictions)  # Collect predictions for all batches


In [ ]:
test_df.head()

In [ ]:
# # Prepare datasets for fine-tuning the AraBART model
# def tokenize_function_test(examples):
#     input_texts = []
#     for question, answer, category in zip(examples['question'], examples['answer'], examples['category']): #examples['predicted_category']
#         input_texts.append(f"category: {category} question: {question}")
#     #model_inputs = tokenizer(input_texts, max_length=max_length_input, truncation=True, padding="max_length")
#     #return model_inputs
#     return input_texts
# test_prepared=test_df.apply(lambda x: tokenize_function_test(x),axis=1)

In [ ]:
# from torch.utils.data import Subset
# from tqdm import tqdm  # For progress tracking

# test_dataset = Dataset.from_pandas(test_df).map(tokenize_function_test, batched=True)

# # Split the test dataset into smaller subsets (e.g., chunks of 100)
# num_samples = len(test_dataset)
# chunk_size = 50  # Adjust based on memory availability
# predictions_list = []

# for i in tqdm(range(0, num_samples, chunk_size)):
#     subset = Subset(test_dataset, range(i, min(i + chunk_size, num_samples)))
    
    
#     predictions = trainer.predict(subset)
#     predictions_list.append(predictions)
    

# # If you want to decode the actual labels as well:
# decoded_preds=[]
# decoded_labels = []
# for pred in predictions_list:
#     decoded_preds.extend(tokenizer.batch_decode(pred.predictions, skip_special_tokens=True))
#     decoded_labels.extend(tokenizer.batch_decode(pred.label_ids, skip_special_tokens=True))

# # Display some predictions and labels for comparison
# for i in range(5):  # Show the first 5 examples
#     print(f"Prediction {i}: {decoded_preds[i]}")
#     print(f"Label {i}: {decoded_labels[i]}")


In [ ]:
import pickle

# Save predictions_list to a file
with open("./models/AraBERT_BiLSTM/Basic_with_preprocessing/train_predictions_list.pkl", "wb") as file:
    pickle.dump(train_predictions_list, file)

with open("./models/AraBERT_BiLSTM/Basic_with_preprocessing/test_predictions_list.pkl", "wb") as file:
    pickle.dump(test_predictions_list, file)

In [ ]:
# # Load predictions_list from a file
# with open("models/predictions_list-moussaKam-AraBART.pkl", "rb") as file:
#     predictions_list = pickle.load(file)

In [ ]:
# import numpy as np
# from nltk.translate.bleu_score import sentence_bleu
# # Test and calculate BLEU score
# chencherry = SmoothingFunction()

# def compute_bleu_score(preds, labels):
#     bleu_scores = []
#     for pred, label in zip(preds, labels):
#         reference = [label.split()]
#         candidate = pred.split()
#         bleu_scores.append(sentence_bleu(reference, candidate, smoothing_function=chencherry.method1))
#     return np.mean(bleu_scores)

# # Compute BLEU score
# bleu_score = compute_bleu_score(train_predictions_list, test_df['answer'])
# print(f"BLEU Score: {bleu_score}")

In [ ]:
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# reference = [["this", "is", "a", "reference", "sentence"]]  # List of lists
# candidate = ["this", "is", "a", "predicted", "sentence"]   # Single list

# score = sentence_bleu(reference, candidate)
# print("BLEU score:", score)

In [ ]:
# from nltk.translate.bleu_score import sentence_bleu

# reference = [["this", "is", "a", "reference", "sentence"]]  # List of lists
# candidate = ["this", "is", "a", "reference", "sentence"]   # Single list

# score = sentence_bleu(reference, candidate)
# print("BLEU score:", score)

In [ ]:
list(train_df['question'][0:5])

In [ ]:
train_predictions_list[0:5]

In [ ]:
list(train_df['answer'][0:5])

In [ ]:
from collections import Counter
import math

def calculate_bleu(predictions, references, max_n=4, weights=[0.25, 0.25, 0.25, 0.25]):
    total_candidate_length = 0
    total_reference_length = 0
    ngram_precisions = []

    # Step 1: Compute n-gram precisions for each n (1 to max_n)
    for n in range(1, max_n + 1):
        ngram_matches = 0
        ngram_total = 0
        
        for prediction, reference in zip(predictions, references):
            # Convert sentences to tokens
            pred_tokens = prediction.split()
            ref_tokens = reference.split()
            
            # Update lengths for brevity penalty calculation
            total_candidate_length += len(pred_tokens)
            total_reference_length += len(ref_tokens)
            
            # Generate n-grams for prediction and reference
            pred_ngrams = [tuple(pred_tokens[i:i+n]) for i in range(len(pred_tokens)-n+1)]
            ref_ngrams = [tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1)]
            
            # Count n-gram matches
            pred_ngram_counts = Counter(pred_ngrams)
            ref_ngram_counts = Counter(ref_ngrams)
            
            # Calculate clipped counts
            for ngram in pred_ngram_counts:
                ngram_matches += min(pred_ngram_counts[ngram], ref_ngram_counts[ngram])
            ngram_total += len(pred_ngrams)
        
        # Precision for current n-gram level
        precision = (ngram_matches / ngram_total) if ngram_total > 0 else 0
        ngram_precisions.append(precision)

    # Step 2: Calculate log-precision sum with weights
    weighted_log_precisions = sum(weights[i] * math.log(ngram_precisions[i]) for i in range(max_n) if ngram_precisions[i] > 0)
    
    # Step 3: Calculate Brevity Penalty (BP)
    if total_candidate_length > total_reference_length:
        BP = 1
    else:
        BP = math.exp(1 - (total_reference_length / total_candidate_length)) if total_candidate_length > 0 else 0

    # Step 4: Compute BLEU score
    bleu_score = BP * math.exp(weighted_log_precisions)

    return bleu_score

# Example usage
pred_list = ["the cat is on the mat", "there is a cat"]
ref_list = ["the cat is on the mat", "there is the cat"]

bleu_score = calculate_bleu(pred_list, ref_list)
print("BLEU Score:", bleu_score)

In [ ]:
train_bleu_score = calculate_bleu(train_predictions_list, train_df['answer'])

In [ ]:
train_bleu_score

In [ ]:
test_bleu_score = calculate_bleu(test_predictions_list, test_df['answer'])

In [ ]:
test_bleu_score

In [ ]:
x=test_df['question'].apply(lambda x: len(x))

In [ ]:
x.max()

In [ ]:
x.min()

In [ ]:
x.idxmin()

In [ ]:
test_df['question'][334415]

In [ ]:
x.mean()

In [ ]:
x=test_df['answer'].apply(lambda x: len(x))

In [ ]:
x.max()

In [ ]:
x.min()

In [ ]:
x=test_df['category'].apply(lambda x: len(x))

In [ ]:
x.max()

In [ ]:
x.min()

In [ ]:
test_df.shape

In [ ]:
test_df.shape[0]/4

In [ ]:
test_dataset

In [ ]:
import sys

size_in_bytes = sys.getsizeof(test_dataset)
print(size_in_bytes)

In [ ]:
type(test_dataset)

In [ ]:
test_dataset[0]